# AI Research Paper Assistant with Explainable Similarity Ranking

**Final capstone project: AI Agentic Systems**

## 1. Project Overview

Upload **one research paper (PDF)**. The system then:

1. **Answers questions about the paper** using RAG (Retrieval-Augmented Generation).
   Answers are grounded in retrieved text and cite page numbers.
2. **Automatically discovers, ranks, and explains similar papers.** It uses
   Semantic Scholar (references, citations, recommendations) and arXiv (LLM-written search queries).
   All scores are **computed in Python** from embeddings and cosine similarity. The LLM only *explains* the scores and never computes them.

**Exactly two agents** (both built with LangChain `create_agent`):

| Agent | Job | Tools |
|---|---|---|
| **Paper RAG Agent** | Answer free-form questions about the uploaded paper | `search_paper` (retriever over the PDF) |
| **Discovery & Ranking Agent** | Find candidate papers, then (via its graph branch) merge, score, rank, explain | `get_semantic_scholar_candidates`, `search_arxiv` |

Everything else (PDF loading, profile extraction, merging, scoring, explanation writing, routing) is a
**plain Python function or LangGraph node**, not an agent.

**How to run (Google Colab):**
1. Add a secret named `GROQ_API_KEY` (or `GOOGLE_API_KEY` for Gemini) using the key icon in the left sidebar, and enable *Notebook access*.
   Optional: `SEMANTIC_SCHOLAR_API_KEY` for higher Semantic Scholar rate limits.
2. `Runtime → Run all`. When asked, upload your PDF (or set `PDF_SOURCE = 'sample'` to use the ReAct paper).

## 2. Architecture

```
                         Uploaded PDF
                              |
                              v
                     [load_pdf]  (pages + page numbers)
                              |
                              v
                     [extract_profile]  (LLM -> structured profile)
                              |
                 +------------+-------------+        <- PARALLEL fan-out
                 |                          |
                 v                          v
         [build_rag_index]          [discovery_agent]   (Agent 2: LLM + tools)
                 |                          |   Semantic Scholar refs / cites / recs
          question given?                   |   arXiv: 4-5 LLM-written queries
           yes /     \ no                   v
              v       END            [merge_dedup]
     [paper_rag_agent]                      |
       (Agent 1)                   any candidates?
              |                     yes /       \ no
              v                        v         END
             END                [score_rank]  (Python: cosine similarity + formula)
                                       |
                                       v
                               [explain_ranking]  (LLM explains, using evidence only)
                                       |
                                       v
                                      END
```

**Routing at START** (`route_request`): on the *first* run the paper has not been processed, so we go to `load_pdf`.
On *later* runs (follow-up questions) the paper is already indexed, so we jump **straight to the Paper RAG Agent**
and skip everything else.

Non-sequential patterns used:
- **Conditional routing** at START, after indexing, and after merging
- **Parallel branches** (RAG indexing and paper discovery run at the same time)

## 3. Dependencies

In [ ]:
!pip install -q "langchain>=1.4" "langgraph>=1.2.11" "langchain-groq>=1.1" "langchain-google-genai>=4.0" \
    "langchain-community>=0.4" "langchain-text-splitters>=1.0" "langchain-huggingface>=1.2" \
    sentence-transformers faiss-cpu pypdf arxiv requests

## 4. Configuration and LLM Setup

All tunable settings live in this one cell. To switch LLM provider, change **only** `LLM_PROVIDER`.
`init_chat_model("provider:model")` builds the correct LangChain chat model class for us.
API keys are read from **Colab Secrets** and are never hard-coded.

In [ ]:
import asyncio
import getpass
import json
import os
import re
import threading
import time
import warnings
from typing import TypedDict

import numpy as np
from langchain.agents import create_agent
from langchain.agents.middleware import ToolCallLimitMiddleware
from langchain.chat_models import init_chat_model
from langchain_core.messages import AIMessage, HumanMessage, SystemMessage, ToolMessage
from langchain_core.tools import tool
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.graph import END, START, StateGraph
from pydantic import BaseModel, Field

warnings.filterwarnings('ignore', category=DeprecationWarning)   # keep demo output readable

# ----------------------------- LLM -----------------------------------
LLM_PROVIDER = 'groq'   # 'groq' or 'gemini'
PROVIDER_SETTINGS = {
    'groq':   {'secret': 'GROQ_API_KEY',   'model': 'groq:llama-3.3-70b-versatile'},
    'gemini': {'secret': 'GOOGLE_API_KEY', 'model': 'google_genai:gemini-3.8-flash'},
}

# ----------------------------- PDF / RAG ------------------------------
PDF_SOURCE = 'upload'                     # 'upload' (Colab file picker) or 'sample'
SAMPLE_PDF_URL = 'https://arxiv.org/pdf/2210.03629'   # ReAct paper, used when PDF_SOURCE='sample'
EMBEDDING_MODEL_NAME = 'sentence-transformers/all-MiniLM-L6-v2'   # free, runs inside Colab
CHUNK_SIZE = 1000
CHUNK_OVERLAP = 150
RAG_TOP_K = 4                             # chunks returned per retrieval

# ----------------------------- Discovery -----------------------------
S2_RELATION_LIMIT = 50                    # max references / citations fetched
S2_RECOMMENDATION_LIMIT = 20
ARXIV_RESULTS_PER_QUERY = 8
MIN_ABSTRACT_CHARS = 50                   # shorter abstracts cannot be compared semantically

# ----------------------------- Ranking --------------------------------
TOP_N_RESULTS = 3
SCORE_WEIGHTS = {                         # agreed formula; semantic similarity dominates
    'semantic_similarity':    0.70,
    'citation_relationship':  0.15,
    'multi_source_agreement': 0.10,
    'topic_overlap':          0.05,
}


def get_secret(name: str, required: bool = True) -> str:
    """Read a secret from Colab Secrets, else an environment variable, else ask (if required)."""
    value = ''
    try:
        from google.colab import userdata
        value = userdata.get(name) or ''
    except Exception:          # not running in Colab, or the secret is not set
        value = os.environ.get(name, '')
    if not value and required:
        value = getpass.getpass(f'Paste your {name}: ')
    if not value and required:
        raise ValueError(f'{name} is missing. Add it to Colab Secrets.')
    return value


settings = PROVIDER_SETTINGS[LLM_PROVIDER]
os.environ[settings['secret']] = get_secret(settings['secret'])
S2_API_KEY = get_secret('SEMANTIC_SCHOLAR_API_KEY', required=False)   # optional

MODEL_NAME = settings['model']
llm = init_chat_model(MODEL_NAME, temperature=0, max_retries=3)
print('LLM configured:', MODEL_NAME)
print('Semantic Scholar API key:', 'yes' if S2_API_KEY else 'no (public rate limit, slower)')

In [ ]:
# Smoke test: one tiny LLM call
print(llm.invoke('Reply with exactly: LLM ready').content)

### Observability: tracing + loop detection

A small tracer of our own. Every graph node, agent run, LLM call, and tool call opens a **span**.
When the top-level run finishes we print the **trace tree** with durations and token counts,
so the execution flow is visible during the presentation.

`LoopDetector` protects the agents:
- **Repetition**: the same (or nearly the same) tool call is made again, so the tool refuses and tells the agent to change approach.
- **Stagnation**: the last few arXiv searches returned nearly the same papers, so the agent is warned to diversify its queries.

In [ ]:
import functools
from contextlib import contextmanager
from contextvars import ContextVar
from dataclasses import dataclass, field


@dataclass
class Span:
    name: str
    kind: str                 # 'run', 'node', 'agent', 'llm', 'tool'
    depth: int
    start: float = field(default_factory=time.time)
    end: float = 0.0
    tokens: int = 0
    notes: list = field(default_factory=list)
    children: list = field(default_factory=list)


_current_span: ContextVar = ContextVar('current_span', default=None)


def total_tokens(span: Span) -> int:
    return span.tokens + sum(total_tokens(child) for child in span.children)


def print_trace(span: Span):
    indent = '    ' * span.depth
    branch = '└─ ' if span.depth else '=== '
    tokens = total_tokens(span)
    token_text = f'  tokens={tokens}' if tokens else ''
    print(f'{indent}{branch}{span.name} [{span.kind}] {span.end - span.start:.2f}s{token_text}')
    for note in span.notes:
        print(f'{indent}      ! {note}')
    for child in span.children:
        print_trace(child)


@contextmanager
def trace_span(name: str, kind: str = 'node'):
    """Open a span, nested under whatever span is currently active."""
    parent = _current_span.get()
    span = Span(name=name, kind=kind, depth=parent.depth + 1 if parent else 0)
    if parent:
        parent.children.append(span)
    token = _current_span.set(span)
    try:
        yield span
    finally:
        span.end = time.time()
        _current_span.reset(token)
        if parent is None:                      # top-level run finished -> print tree
            print('\n' + '-' * 70)
            print_trace(span)
            print('-' * 70 + '\n')


def traced(name: str, kind: str = 'node'):
    """Decorator version of trace_span for sync or async functions."""
    def decorator(func):
        if asyncio.iscoroutinefunction(func):
            @functools.wraps(func)
            async def async_wrapper(*args, **kwargs):
                with trace_span(name, kind):
                    return await func(*args, **kwargs)
            return async_wrapper

        @functools.wraps(func)
        def sync_wrapper(*args, **kwargs):
            with trace_span(name, kind):
                return func(*args, **kwargs)
        return sync_wrapper
    return decorator


def record_tokens(message) -> int:
    """Add an LLM message's token usage to the current span."""
    usage = getattr(message, 'usage_metadata', None) or {}
    tokens = usage.get('total_tokens', 0)
    span = _current_span.get()
    if span:
        span.tokens += tokens
    return tokens


def trace_note(text: str):
    """Attach a visible warning/note to the current span (and print it immediately)."""
    print('  [trace]', text)
    span = _current_span.get()
    if span:
        span.notes.append(text)


class LoopDetector:
    """Detects repeated tool calls (exact / fuzzy) and stagnating tool outputs."""

    def __init__(self, exact_threshold=2, fuzzy_threshold=0.8, stagnation_window=3):
        self.exact_threshold = exact_threshold
        self.fuzzy_threshold = fuzzy_threshold
        self.stagnation_window = stagnation_window
        self.reset()

    def reset(self):
        self.tool_history = []        # (tool_name, normalized_input)
        self.output_history = []

    @staticmethod
    def jaccard(text_a: str, text_b: str) -> float:
        words_a, words_b = set(text_a.lower().split()), set(text_b.lower().split())
        if not words_a or not words_b:
            return 0.0
        return len(words_a & words_b) / len(words_a | words_b)

    def check_tool_call(self, tool_name: str, tool_input: str) -> str:
        """Return a warning message if this call repeats earlier ones, else ''."""
        current = (tool_name, tool_input.strip().lower())
        exact = sum(1 for past in self.tool_history if past == current)
        fuzzy = sum(1 for name, past_input in self.tool_history[-5:]
                    if name == tool_name and self.jaccard(past_input, current[1]) >= self.fuzzy_threshold)
        self.tool_history.append(current)
        if exact >= self.exact_threshold:
            return f"Repetition detected: '{tool_name}' was already called {exact} times with identical input. Change your approach."
        if fuzzy >= self.exact_threshold:
            return f"Repetition detected: '{tool_name}' was called with near-identical input {fuzzy} times. Try a clearly different input."
        return ''

    def check_output_stagnation(self, output: str) -> str:
        """Return a warning if the last few outputs are nearly identical, else ''."""
        self.output_history.append(output)
        recent = self.output_history[-self.stagnation_window:]
        if len(recent) < self.stagnation_window:
            return ''
        pairs = [self.jaccard(recent[i], recent[j]) for i in range(len(recent)) for j in range(i + 1, len(recent))]
        average = sum(pairs) / len(pairs)
        if average >= self.fuzzy_threshold:
            return (f'Stagnation detected: the last {self.stagnation_window} results are {average:.0%} similar. '
                    'Your next query must cover a clearly different aspect of the paper.')
        return ''


rag_loop_detector = LoopDetector()
discovery_loop_detector = LoopDetector()
print('Tracing and LoopDetector ready.')

## 5. PDF Loading

`PyPDFLoader` returns **one LangChain `Document` per page**, with the page index in its metadata.
We keep a human-readable `page_number` (starting at 1) so RAG answers can cite pages.
We also look for an **arXiv ID or DOI** on the first page, which helps Semantic Scholar find the exact paper.

In [ ]:
import requests
from langchain_community.document_loaders import PyPDFLoader


def get_pdf_path() -> str:
    """Ask the user to upload a PDF in Colab, or download the sample paper."""
    if PDF_SOURCE == 'sample':
        path = 'sample_paper.pdf'
        response = requests.get(SAMPLE_PDF_URL, timeout=60)
        response.raise_for_status()
        with open(path, 'wb') as f:
            f.write(response.content)
        return path

    from google.colab import files
    uploaded = files.upload()                       # opens the file picker
    pdf_names = [name for name in uploaded if name.lower().endswith('.pdf')]
    if not pdf_names:
        raise ValueError('Please upload a .pdf file.')
    return pdf_names[0]


def load_pdf_pages(pdf_path: str) -> list:
    """Load the PDF as one Document per page, with clean text and a 1-based page number."""
    pages = PyPDFLoader(pdf_path).load()
    cleaned_pages = []
    for page in pages:
        text = re.sub(r'[ \t]+', ' ', page.page_content).strip()
        if not text:
            continue                                # skip empty / image-only pages
        page.page_content = text
        page.metadata['page_number'] = page.metadata.get('page', 0) + 1
        cleaned_pages.append(page)
    if not cleaned_pages:
        raise ValueError('No text could be extracted. Is this a scanned (image-only) PDF?')
    return cleaned_pages


def find_paper_identifiers(pages: list) -> dict:
    """Look for an arXiv ID or DOI on the first page (not in the reference list)."""
    first_page = pages[0].page_content
    arxiv_match = re.search(r'arXiv:\s*(\d{4}\.\d{4,5})', first_page)
    doi_match = re.search(r'\b(10\.\d{4,9}/[^\s"<>]+)', first_page)
    return {
        'arxiv_id': arxiv_match.group(1) if arxiv_match else '',
        'doi': doi_match.group(1).rstrip('.,;)') if doi_match else '',
    }


PDF_PATH = get_pdf_path()
_preview_pages = load_pdf_pages(PDF_PATH)
print(f'Loaded {PDF_PATH}: {len(_preview_pages)} pages with text')
print('Identifiers found on page 1:', find_paper_identifiers(_preview_pages))
print('\nFirst 300 characters:\n', _preview_pages[0].page_content[:300])

## 6. Paper Profile Extraction

The LLM reads the beginning of the paper (title, abstract, introduction) and fills a **Pydantic schema**.
`with_structured_output` forces the LLM to return exactly these fields, so the output is always valid.

The profile is used by **both** sides of the system:
- Agent 2 uses the problem / method / domain / concepts to write arXiv search queries.
- Ranking uses the title + abstract (semantic similarity) and keywords + concepts (topic overlap).

In [ ]:
class PaperProfile(BaseModel):
    title: str = Field(description='Exact title of the paper')
    abstract: str = Field(description='The abstract, copied as written in the paper')
    research_problem: str = Field(description='The problem the paper tries to solve, 1-2 sentences')
    methodology: str = Field(description='The method / approach used, 1-2 sentences')
    application_domain: str = Field(description='Field or application area, a few words')
    important_concepts: list[str] = Field(description='5-8 key technical concepts')
    keywords: list[str] = Field(description='5-8 short keywords (1-3 words each)')


PROFILE_PROMPT = """You extract a structured profile of a research paper.
Use ONLY the paper text provided. Do not add outside knowledge.
Copy the title and abstract as written. Keep the other fields short and specific."""

PROFILE_INPUT_CHARS = 15000      # title + abstract + introduction are at the start of the paper


def extract_paper_profile(pages: list) -> dict:
    paper_start = '\n\n'.join(page.page_content for page in pages)[:PROFILE_INPUT_CHARS]
    structured_llm = llm.with_structured_output(PaperProfile, include_raw=True)
    with trace_span('llm:extract_profile', 'llm'):
        result = structured_llm.invoke([
            SystemMessage(PROFILE_PROMPT),
            HumanMessage(f'Paper text:\n\n{paper_start}'),
        ])
        record_tokens(result['raw'])
    if result['parsed'] is None:
        raise ValueError(f"Profile extraction failed: {result.get('parsing_error')}")
    return result['parsed'].model_dump()


print('extract_paper_profile() defined. It runs inside the graph (node: extract_profile).')

## 7. RAG Setup

**PDF pages → chunks → embeddings → FAISS vector store.**

- `RecursiveCharacterTextSplitter` cuts pages into ~1000-character chunks with overlap.
  Each chunk **inherits its page's metadata**, so page numbers survive.
- `all-MiniLM-L6-v2` is a free sentence-embedding model that runs inside Colab, with no API key needed.
  We normalize embeddings, so the vector distance is equivalent to **cosine similarity**.
- The same embedding model is reused later for paper-to-paper similarity.

In [ ]:
from langchain_community.vectorstores import FAISS
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter

embedding_model = HuggingFaceEmbeddings(
    model_name=EMBEDDING_MODEL_NAME,
    encode_kwargs={'normalize_embeddings': True},
)
text_splitter = RecursiveCharacterTextSplitter(chunk_size=CHUNK_SIZE, chunk_overlap=CHUNK_OVERLAP)

# The vector store for the current paper. The search_paper tool reads it.
PAPER_INDEX = {'vector_store': None, 'num_chunks': 0}


def build_rag_index(pages: list) -> int:
    chunks = text_splitter.split_documents(pages)
    PAPER_INDEX['vector_store'] = FAISS.from_documents(chunks, embedding_model)
    PAPER_INDEX['num_chunks'] = len(chunks)
    return len(chunks)


def retrieve_chunks(query: str, k: int = RAG_TOP_K) -> list:
    if PAPER_INDEX['vector_store'] is None:
        raise RuntimeError('RAG index not built yet.')
    return PAPER_INDEX['vector_store'].similarity_search(query, k=k)


# Quick check on the loaded PDF (the graph rebuilds it during the real run)
print('Chunks:', build_rag_index(_preview_pages))
for doc in retrieve_chunks('main contribution of the paper', k=2):
    print(f"[page {doc.metadata['page_number']}] {doc.page_content[:150]}...")

## 8. Semantic Scholar Client

Three REST endpoints give us three kinds of candidates:

| Relation | Endpoint | Meaning |
|---|---|---|
| references | `/graph/v1/paper/{id}/references` | papers the uploaded paper **cites** |
| citations | `/graph/v1/paper/{id}/citations` | papers that **cite** the uploaded paper |
| recommendations | `/recommendations/v1/papers/forpaper/{id}` | papers Semantic Scholar thinks are related |

First we must find the uploaded paper on Semantic Scholar: by arXiv ID, then DOI, then title match.
Without an API key the public rate limit is shared, so `s2_get` **retries on HTTP 429** with increasing waits.

Every candidate from every source is converted to the **same dictionary format** by `make_candidate`.

In [ ]:
S2_BASE_URL = 'https://api.semanticscholar.org'
S2_FIELDS = 'paperId,title,abstract,authors,year,externalIds,url'


def s2_get(path: str, params: dict | None = None, max_attempts: int = 5):
    """GET a Semantic Scholar endpoint. Retries on 429/5xx. Returns parsed JSON or None."""
    headers = {'x-api-key': S2_API_KEY} if S2_API_KEY else {}
    for attempt in range(max_attempts):
        try:
            response = requests.get(S2_BASE_URL + path, params=params, headers=headers, timeout=30)
        except requests.RequestException as error:
            print(f'  Semantic Scholar network error: {error}')
            time.sleep(2 ** attempt)
            continue
        if response.status_code == 200:
            return response.json()
        if response.status_code == 404:
            return None
        if response.status_code == 429 or response.status_code >= 500:
            time.sleep(2 ** attempt + 1)    # 2s, 3s, 5s, 9s, 17s
            continue
        print(f'  Semantic Scholar HTTP {response.status_code}: {response.text[:200]}')
        return None
    print(f'  Semantic Scholar: giving up on {path} after {max_attempts} attempts')
    return None


def clean_doi(doi: str) -> str:
    return (doi or '').strip().lower()


def make_candidate(title, abstract, authors, year, url, source,
                   s2_id='', arxiv_id='', doi='', citation_relationship=''):
    """The single candidate format used by every discovery source."""
    doi = clean_doi(doi)
    if doi.startswith('10.48550/arxiv.') and not arxiv_id:     # arXiv's own DOI -> arXiv ID
        arxiv_id = doi.split('arxiv.', 1)[1]
    return {
        's2_id': s2_id or '',
        'arxiv_id': re.sub(r'v\d+$', '', arxiv_id or ''),     # drop version suffix (v1, v2 ...)
        'doi': doi,
        'title': ' '.join((title or '').split()),
        'abstract': ' '.join((abstract or '').split()),
        'authors': authors or [],
        'year': year,
        'url': url or '',
        'found_in': [source],
        'citation_relationship': citation_relationship,     # '' when there is none
    }


def candidate_from_s2(paper: dict, source: str, citation_relationship: str = '') -> dict:
    external_ids = paper.get('externalIds') or {}
    return make_candidate(
        title=paper.get('title'), abstract=paper.get('abstract'),
        authors=[a.get('name', '') for a in paper.get('authors') or []],
        year=paper.get('year'), url=paper.get('url'), source=source,
        s2_id=paper.get('paperId') or '', arxiv_id=external_ids.get('ArXiv', ''),
        doi=external_ids.get('DOI', ''), citation_relationship=citation_relationship,
    )


def find_uploaded_paper_on_s2(title: str, arxiv_id: str = '', doi: str = ''):
    """Locate the uploaded paper: arXiv ID first, then DOI, then best title match."""
    if arxiv_id:
        paper = s2_get(f'/graph/v1/paper/arXiv:{arxiv_id}', {'fields': S2_FIELDS})
        if paper:
            return paper
    if doi:
        paper = s2_get(f'/graph/v1/paper/DOI:{doi}', {'fields': S2_FIELDS})
        if paper:
            return paper
    if title:
        result = s2_get('/graph/v1/paper/search/match', {'query': title, 'fields': S2_FIELDS})
        if result and result.get('data'):
            return result['data'][0]
    return None


def fetch_s2_related_papers(paper_id: str) -> list:
    """References + citations + recommendations as candidate dicts."""
    candidates = []

    refs = s2_get(f'/graph/v1/paper/{paper_id}/references', {'fields': S2_FIELDS, 'limit': S2_RELATION_LIMIT})
    for item in (refs or {}).get('data') or []:
        if item.get('citedPaper', {}).get('paperId'):
            candidates.append(candidate_from_s2(item['citedPaper'], 'semantic_scholar_reference',
                                                'uploaded paper cites this paper'))

    cites = s2_get(f'/graph/v1/paper/{paper_id}/citations', {'fields': S2_FIELDS, 'limit': S2_RELATION_LIMIT})
    for item in (cites or {}).get('data') or []:
        if item.get('citingPaper', {}).get('paperId'):
            candidates.append(candidate_from_s2(item['citingPaper'], 'semantic_scholar_citation',
                                                'this paper cites the uploaded paper'))

    recs = s2_get(f'/recommendations/v1/papers/forpaper/{paper_id}',
                  {'fields': S2_FIELDS, 'limit': S2_RECOMMENDATION_LIMIT})
    for paper in (recs or {}).get('recommendedPapers') or []:
        candidates.append(candidate_from_s2(paper, 'semantic_scholar_recommendation'))

    return candidates


print('Semantic Scholar client ready.')

## 9. arXiv Client

arXiv is the **complementary** source. It can find relevant papers that have *no citation link* to the uploaded paper.
The *queries* are written by Agent 2 (the LLM) from the paper profile. This function only executes one query.
The `arxiv` library already waits 3 seconds between requests (arXiv's usage policy), and a lock makes
parallel tool calls run one at a time.

In [ ]:
import arxiv

arxiv_client = arxiv.Client(page_size=ARXIV_RESULTS_PER_QUERY, delay_seconds=3, num_retries=3)
arxiv_lock = threading.Lock()


def search_arxiv_papers(query: str, max_results: int = ARXIV_RESULTS_PER_QUERY) -> list:
    """Run one arXiv search and return candidate dicts (source label is set later)."""
    search = arxiv.Search(query=query, max_results=max_results, sort_by=arxiv.SortCriterion.Relevance)
    with arxiv_lock:
        results = list(arxiv_client.results(search))
    return [
        make_candidate(
            title=result.title, abstract=result.summary,
            authors=[author.name for author in result.authors],
            year=result.published.year if result.published else None,
            url=re.sub(r'v\d+$', '', result.entry_id), source='arxiv',
            arxiv_id=result.get_short_id(), doi=result.doi or '',
        )
        for result in results
    ]


print('arXiv client ready.')

## 10. Candidate Merge & Deduplication

The same paper often arrives several times (e.g. as a Semantic Scholar recommendation **and** an arXiv result).
We merge duplicates using identifiers in **priority order**:

1. Semantic Scholar paper ID → 2. arXiv ID → 3. DOI → 4. normalized title

When two records match, we **combine** them: union of `found_in`, keep any citation relationship,
and fill missing fields (e.g. an abstract from arXiv when Semantic Scholar had none).

Then we **exclude**: the uploaded paper itself, and papers without a title or a usable abstract
(no abstract means no semantic similarity can be computed).

In [ ]:
def normalize_title(title: str) -> str:
    return ' '.join(re.sub(r'[^a-z0-9 ]', ' ', (title or '').lower()).split())


def identity_keys(paper: dict) -> list:
    """Identifiers of a paper, in deduplication priority order."""
    keys = []
    if paper.get('s2_id'):
        keys.append(('s2', paper['s2_id']))
    if paper.get('arxiv_id'):
        keys.append(('arxiv', paper['arxiv_id']))
    if paper.get('doi'):
        keys.append(('doi', paper['doi']))
    if normalize_title(paper.get('title')):
        keys.append(('title', normalize_title(paper['title'])))
    return keys


def combine_records(existing: dict, new: dict):
    """Merge `new` into `existing` (same paper found twice)."""
    for source in new['found_in']:
        if source not in existing['found_in']:
            existing['found_in'].append(source)
    for key in ['s2_id', 'arxiv_id', 'doi', 'url', 'year', 'citation_relationship']:
        if not existing.get(key) and new.get(key):
            existing[key] = new[key]
    if len(new.get('abstract', '')) > len(existing.get('abstract', '')):
        existing['abstract'] = new['abstract']
    if not existing.get('authors') and new.get('authors'):
        existing['authors'] = new['authors']


def merge_and_deduplicate(raw_candidates: list, uploaded_paper: dict) -> tuple[list, dict]:
    uploaded_keys = set(identity_keys(uploaded_paper))
    merged = []          # list of unique papers
    key_to_index = {}    # identifier -> position in `merged`
    stats = {'raw': len(raw_candidates), 'duplicates_merged': 0,
             'excluded_uploaded_paper': 0, 'excluded_missing_info': 0}

    for candidate in raw_candidates:
        keys = identity_keys(candidate)
        if uploaded_keys & set(keys):
            stats['excluded_uploaded_paper'] += 1
            continue
        match = next((key_to_index[key] for key in keys if key in key_to_index), None)  # priority order
        if match is None:
            merged.append({**candidate, 'found_in': list(candidate['found_in'])})
            match = len(merged) - 1
        else:
            combine_records(merged[match], candidate)
            stats['duplicates_merged'] += 1
        for key in identity_keys(merged[match]):
            key_to_index.setdefault(key, match)

    usable = [paper for paper in merged
              if paper['title'] and len(paper.get('abstract', '')) >= MIN_ABSTRACT_CHARS]
    stats['excluded_missing_info'] = len(merged) - len(usable)
    stats['unique_usable'] = len(usable)
    return usable, stats


# --- Offline unit test with fake data (no API calls) ---
_uploaded = make_candidate('My Paper', 'x' * 60, [], 2024, '', 'upload', arxiv_id='2401.00001')
_raw = [
    make_candidate('Paper A', 'abstract ' * 10, ['Ann'], 2023, 'u1', 'semantic_scholar_recommendation', s2_id='S1', arxiv_id='2301.11111'),
    make_candidate('Paper A', 'abstract ' * 10, ['Ann'], 2023, 'u2', 'arxiv_query_2', arxiv_id='2301.11111v2'),   # same arXiv ID
    make_candidate('PAPER  b!', 'abstract ' * 10, ['Bob'], 2022, 'u3', 'semantic_scholar_reference', s2_id='S2',
                   citation_relationship='uploaded paper cites this paper'),
    make_candidate('Paper B', 'abstract ' * 10, ['Bob'], 2022, 'u4', 'arxiv_query_1'),                            # same title
    make_candidate('Paper C', '', ['Cy'], 2021, 'u5', 'semantic_scholar_citation', s2_id='S3'),                   # no abstract
    make_candidate('My Paper', 'x' * 60, [], 2024, '', 'arxiv_query_1', arxiv_id='2401.00001'),                   # uploaded paper
]
_merged, _stats = merge_and_deduplicate(_raw, _uploaded)
print(_stats)
for p in _merged:
    print(f"  {p['title']:<10} found_in={p['found_in']}  citation='{p['citation_relationship']}'")
assert _stats == {'raw': 6, 'duplicates_merged': 2, 'excluded_uploaded_paper': 1,
                  'excluded_missing_info': 1, 'unique_usable': 2}
print('Merge & dedup unit test passed.')

## 11. Similarity Ranking (pure Python, no LLM)

**Overall Relatedness Score** (0–100):

```
Overall = 100 × ( 0.70 × semantic_similarity
                + 0.15 × citation_relationship
                + 0.10 × multi_source_agreement
                + 0.05 × topic_overlap )
```

| Component | How it is computed | Range |
|---|---|---|
| `semantic_similarity` | cosine similarity of embeddings of *title + abstract* (uploaded vs candidate), clipped at 0 | 0–1 |
| `citation_relationship` | 1.0 if the uploaded paper cites it or it cites the uploaded paper, else 0.0 | 0 or 1 |
| `multi_source_agreement` | number of distinct discovery **methods** (S2 reference / S2 citation / S2 recommendation / arXiv search; several arXiv queries count as one method): 1 → 0.0, 2 → 0.5, 3+ → 1.0 | 0–1 |
| `topic_overlap` | fraction of the uploaded paper's keywords + concepts that appear in the candidate's title/abstract | 0–1 |

Semantic similarity carries 70% of the weight, so it dominates.
These are **relatedness** scores, **not** accuracy or probability.

In [ ]:
def discovery_methods(found_in: list) -> set:
    """Group source labels into methods: all arXiv queries count as one method."""
    return {'arxiv_search' if source.startswith('arxiv') else source for source in found_in}


def multi_source_agreement(found_in: list) -> float:
    return min(1.0, (len(discovery_methods(found_in)) - 1) / 2)


def shared_topics(profile: dict, candidate_text: str) -> tuple[list, list]:
    """Which of the uploaded paper's keywords/concepts appear in the candidate text?"""
    terms = list(dict.fromkeys(t.lower().strip() for t in profile['keywords'] + profile['important_concepts'] if t.strip()))
    text = candidate_text.lower()
    words = set(re.findall(r'[a-z0-9]+', text))
    shared = []
    for term in terms:
        term_words = [w for w in re.findall(r'[a-z0-9]+', term) if len(w) > 2]
        if term in text or (term_words and all(w in words or w.rstrip('s') in words for w in term_words)):
            shared.append(term)
    return shared, terms


def cosine_similarities(query_text: str, texts: list) -> np.ndarray:
    """Embeddings are L2-normalized, so the dot product IS cosine similarity."""
    query_vector = np.array(embedding_model.embed_query(query_text))
    matrix = np.array(embedding_model.embed_documents(texts))
    return matrix @ query_vector


def overall_relatedness(components: dict) -> float:
    return round(100 * sum(SCORE_WEIGHTS[name] * value for name, value in components.items()), 1)


def score_and_rank(uploaded_text: str, profile: dict, candidates: list) -> list:
    texts = [f"{c['title']}. {c['abstract']}" for c in candidates]
    similarities = cosine_similarities(uploaded_text, texts)
    scored = []
    for candidate, text, similarity in zip(candidates, texts, similarities):
        shared, terms = shared_topics(profile, text)
        components = {
            'semantic_similarity': round(float(max(0.0, similarity)), 4),
            'citation_relationship': 1.0 if candidate['citation_relationship'] else 0.0,
            'multi_source_agreement': multi_source_agreement(candidate['found_in']),
            'topic_overlap': round(len(shared) / len(terms), 4) if terms else 0.0,
        }
        scored.append({**candidate, 'components': components, 'shared_keywords': shared,
                       'overall_relatedness': overall_relatedness(components)})
    scored.sort(key=lambda paper: paper['overall_relatedness'], reverse=True)
    for rank, paper in enumerate(scored, start=1):
        paper['rank'] = rank
    return scored


# --- Tiny worked example of the formula ---
_example = {'semantic_similarity': 0.80, 'citation_relationship': 1.0, 'multi_source_agreement': 0.5, 'topic_overlap': 0.4}
print('Worked example:', _example)
print('  100 × (0.70×0.80 + 0.15×1.0 + 0.10×0.5 + 0.05×0.4) =', overall_relatedness(_example))
assert overall_relatedness(_example) == 78.0
print('Agreement check: 1 method →', multi_source_agreement(['arxiv_query_1', 'arxiv_query_3']),
      '| 2 methods →', multi_source_agreement(['semantic_scholar_citation', 'arxiv_query_1']),
      '| 3 methods →', multi_source_agreement(['semantic_scholar_citation', 'semantic_scholar_recommendation', 'arxiv_query_1']))

## 12. Ranking Explanation (LLM explains, Python scores)

For each top paper we build a **structured evidence dictionary** from the ranking process
and give it to the LLM with a strict instruction: *explain using only the supplied evidence*.
The LLM never sees a request to compute or adjust a number.

In [ ]:
EXPLANATION_PROMPT = """You explain why a candidate paper was ranked as related to an uploaded paper.
Explain why this paper ranked highly using ONLY the supplied evidence. Do not invent additional reasons.
- You may quote the supplied scores, but never calculate, change, or invent any number.
- Compare problem / method / domain only with what the candidate's abstract actually says.
- If the evidence shows a weak point (e.g. no citation link), you may mention it briefly.
Return 2-4 short markdown bullet points, nothing else."""


def build_ranking_evidence(paper: dict, profile: dict) -> dict:
    components = paper['components']
    return {
        'candidate_title': paper['title'],
        'candidate_abstract': paper['abstract'][:1200],
        'semantic_similarity': components['semantic_similarity'],
        'overall_relatedness_score': paper['overall_relatedness'],
        'citation_relationship': paper['citation_relationship'] or 'none',
        'found_in': paper['found_in'],
        'number_of_discovery_methods': len(discovery_methods(paper['found_in'])),
        'shared_keywords': paper['shared_keywords'],
        'original_title': profile['title'],
        'original_problem': profile['research_problem'],
        'original_method': profile['methodology'],
        'original_domain': profile['application_domain'],
    }


def explain_ranking(paper: dict, profile: dict) -> str:
    evidence = build_ranking_evidence(paper, profile)
    with trace_span(f"llm:explain #{paper['rank']}", 'llm'):
        response = llm.invoke([
            SystemMessage(EXPLANATION_PROMPT),
            HumanMessage('Evidence:\n' + json.dumps(evidence, indent=2)),
        ])
        record_tokens(response)
    return message_text(response).strip()


def message_text(message) -> str:
    """Message content as plain text (some providers return a list of content parts)."""
    content = message.content
    if isinstance(content, str):
        return content
    return ''.join(part.get('text', '') if isinstance(part, dict) else str(part) for part in content)


print('Explanation step ready.')

## 13. Shared Agent Runner

`run_agent` invokes any of our two agents, records its token usage on the trace, and returns the final answer
plus the **new** messages of this run (the RAG agent has memory, so older turns are skipped when counting).

In [ ]:
async def run_agent(agent, user_message: str, thread_id: str | None = None, max_steps: int = 10) -> dict:
    config = {'recursion_limit': 2 * max_steps + 1}
    if thread_id:
        config['configurable'] = {'thread_id': thread_id}
    with trace_span(f'agent:{agent.name}', 'agent') as span:
        result = await agent.ainvoke({'messages': [HumanMessage(user_message)]}, config=config)
        messages = result['messages']
        last_human = max(i for i, m in enumerate(messages) if isinstance(m, HumanMessage))
        new_messages = messages[last_human:]                       # only this run's messages
        for message in new_messages:
            if isinstance(message, AIMessage):
                record_tokens(message)
        tool_calls = sum(len(m.tool_calls) for m in new_messages if isinstance(m, AIMessage))
        span.notes.append(f'{tool_calls} tool calls, {len(new_messages)} messages')
    return {'answer': message_text(messages[-1]), 'messages': new_messages}

## 14. Agent 1: Paper RAG Agent

- **Tool:** `search_paper`, a retriever over the FAISS index. It returns chunks labelled with page numbers.
- **Grounding rules** in the system prompt: always search first, answer only from retrieved text,
  cite pages, and clearly say when the answer is not in the paper.
- **Memory:** an `InMemorySaver` checkpointer + a `thread_id` per paper, so follow-up questions work
  ("explain that more simply").
- **Reliability:** `ToolCallLimitMiddleware` (max 4 searches per question) + `LoopDetector` repetition check.

In [ ]:
@tool
def search_paper(query: str) -> str:
    """Search the uploaded research paper. Returns the most relevant passages with their page numbers."""
    with trace_span('tool:search_paper', 'tool'):
        warning = rag_loop_detector.check_tool_call('search_paper', query)
        if warning:
            trace_note(warning)
            return warning
        chunks = retrieve_chunks(query)
        if not chunks:
            return 'No relevant passages found.'
        return '\n\n'.join(f"[Page {doc.metadata['page_number']}]\n{doc.page_content}" for doc in chunks)


PAPER_RAG_PROMPT = """You are the Paper RAG Agent. You answer questions about ONE uploaded research paper.

Rules:
1. Always call search_paper before answering. For broad questions (e.g. a summary), search 2-3 aspects
   such as objective, method, results.
2. Answer ONLY from the retrieved passages. Never use outside knowledge about the paper.
3. Cite the page for each claim, like (p. 3).
4. If the passages do not contain the answer, reply: "I could not find this in the uploaded paper."
   You may add what related information the paper does contain.
5. Be concise and clear."""

paper_rag_agent = create_agent(
    model=llm,
    tools=[search_paper],
    system_prompt=PAPER_RAG_PROMPT,
    middleware=[ToolCallLimitMiddleware(run_limit=4, exit_behavior='continue')],
    checkpointer=InMemorySaver(),
    name='paper_rag_agent',
)
print('Agent 1 created:', paper_rag_agent.name)

## 15. Agent 2: Paper Discovery & Ranking Agent

This agent is **not a chatbot**. It starts automatically when a paper is processed.

The **agent part** (LLM + tools) decides *how to search*:
1. calls `get_semantic_scholar_candidates` with the paper's title / arXiv ID / DOI
2. writes **4–5 focused arXiv queries** (problem, method, domain, problem+method, concepts) and calls `search_arxiv` for each

Its **graph branch** then finishes the job deterministically: `merge_dedup → score_rank → explain_ranking`.

Tools return a *short summary* to the LLM (to save tokens), while the full candidate lists travel as a
tool **artifact** (`response_format='content_and_artifact'`). The graph node collects these artifacts
after the agent finishes. This means the LLM never has to copy paper data, and it cannot alter it.

In [ ]:
@tool(response_format='content_and_artifact')
def get_semantic_scholar_candidates(title: str, arxiv_id: str = '', doi: str = '') -> tuple[str, dict]:
    """Find the uploaded paper on Semantic Scholar and fetch its references, citations, and recommended
    related papers. Call this exactly once, with the paper title and any known arXiv ID or DOI."""
    with trace_span('tool:get_semantic_scholar_candidates', 'tool'):
        warning = discovery_loop_detector.check_tool_call('get_semantic_scholar_candidates', title)
        if warning:
            trace_note(warning)
            return warning, {}
        uploaded = find_uploaded_paper_on_s2(title, arxiv_id, doi)
        if not uploaded:
            return 'The paper was not found on Semantic Scholar. Rely on arXiv search.', {}
        candidates = fetch_s2_related_papers(uploaded['paperId'])
        counts = {}
        for c in candidates:
            counts[c['found_in'][0]] = counts.get(c['found_in'][0], 0) + 1
        summary = (f"Found the paper on Semantic Scholar: '{uploaded.get('title')}'. "
                   f'Collected {len(candidates)} candidates: {counts}.')
        return summary, {'uploaded_paper': uploaded, 'candidates': candidates}


@tool(response_format='content_and_artifact')
def search_arxiv(query: str) -> tuple[str, dict]:
    """Search arXiv with ONE focused keyword query (3-8 words, no quotes or boolean operators).
    Returns the titles found."""
    with trace_span('tool:search_arxiv', 'tool'):
        warning = discovery_loop_detector.check_tool_call('search_arxiv', query)
        if warning:
            trace_note(warning)
            return warning, {}
        try:
            candidates = search_arxiv_papers(query)
        except Exception as error:
            return f'arXiv search failed ({error}). Try a different query.', {}
        titles = [c['title'] for c in candidates]
        summary = f"arXiv query '{query}' returned {len(titles)} papers:\n" + '\n'.join(f'- {t}' for t in titles)
        stagnation = discovery_loop_detector.check_output_stagnation(' '.join(titles))
        if stagnation:
            trace_note(stagnation)
            summary += '\n\nWARNING: ' + stagnation
        return summary, {'query': query, 'candidates': candidates}


DISCOVERY_PROMPT = """You are the Paper Discovery & Ranking Agent. You collect candidate papers related to an
uploaded research paper. You do NOT rank or score papers: ranking is computed in Python afterwards.

Steps:
1. Call get_semantic_scholar_candidates ONCE with the paper title (and the arXiv ID / DOI if given).
2. Write 5 focused arXiv search queries (3-8 keywords each, no quotes, no AND/OR) covering:
   (a) the research problem, (b) the methodology, (c) the application domain,
   (d) problem + methodology combined, (e) the most important technical concepts.
   Call search_arxiv once per query. Each query must be clearly different.
3. Finish with a short summary: the queries you used and roughly how many candidates each source gave."""

discovery_agent = create_agent(
    model=llm,
    tools=[get_semantic_scholar_candidates, search_arxiv],
    system_prompt=DISCOVERY_PROMPT,
    middleware=[ToolCallLimitMiddleware(run_limit=8, exit_behavior='continue')],
    name='discovery_ranking_agent',
)

AGENTS = [paper_rag_agent, discovery_agent]      # exactly two agents
print('Agent 2 created:', discovery_agent.name)
print('Total agents:', len(AGENTS))

## 16. LangGraph State and Nodes

`PaperState` is the shared memory of the graph. Each node reads what it needs and returns **only the keys it updates**.
Parallel branches write **different** keys, so they never conflict.

In [ ]:
class PaperState(TypedDict, total=False):
    pdf_path: str
    question: str               # optional: a question for the Paper RAG Agent
    pages: list                 # Documents, one per page
    identifiers: dict           # arXiv ID / DOI found in the PDF
    profile: dict               # structured paper profile
    rag_ready: bool
    uploaded_s2_paper: dict     # the uploaded paper as found on Semantic Scholar
    raw_candidates: list
    discovery_summary: str
    candidates: list            # merged + deduplicated
    merge_stats: dict
    ranked_papers: list         # scored + sorted (+ explanations for the top N)
    answer: str


@traced('load_pdf')
async def load_pdf_node(state: PaperState) -> dict:
    pages = load_pdf_pages(state['pdf_path'])
    return {'pages': pages, 'identifiers': find_paper_identifiers(pages)}


@traced('extract_profile')
async def extract_profile_node(state: PaperState) -> dict:
    return {'profile': extract_paper_profile(state['pages'])}


@traced('build_rag_index')
async def build_rag_index_node(state: PaperState) -> dict:
    num_chunks = await asyncio.to_thread(build_rag_index, state['pages'])
    trace_note(f'{num_chunks} chunks indexed')
    return {'rag_ready': True}


@traced('discovery_agent')
async def discovery_agent_node(state: PaperState) -> dict:
    discovery_loop_detector.reset()
    profile, ids = state['profile'], state['identifiers']
    task = (
        'Find candidate papers related to this uploaded paper.\n\n'
        f"Title: {profile['title']}\n"
        f"arXiv ID: {ids.get('arxiv_id') or 'unknown'}\nDOI: {ids.get('doi') or 'unknown'}\n"
        f"Research problem: {profile['research_problem']}\n"
        f"Methodology: {profile['methodology']}\n"
        f"Application domain: {profile['application_domain']}\n"
        f"Important concepts: {', '.join(profile['important_concepts'])}\n"
        f"Keywords: {', '.join(profile['keywords'])}"
    )
    result = await run_agent(discovery_agent, task, max_steps=12)

    # Collect the full candidate lists from the tool artifacts
    raw_candidates, uploaded_s2_paper, arxiv_query_number = [], {}, 0
    for message in result['messages']:
        if not isinstance(message, ToolMessage) or not message.artifact:
            continue
        artifact = message.artifact
        if message.name == 'get_semantic_scholar_candidates':
            uploaded_s2_paper = artifact.get('uploaded_paper', {})
            raw_candidates += artifact.get('candidates', [])
        elif message.name == 'search_arxiv':
            arxiv_query_number += 1
            for candidate in artifact.get('candidates', []):
                candidate['found_in'] = [f'arxiv_query_{arxiv_query_number}']
                raw_candidates.append(candidate)
    trace_note(f'{len(raw_candidates)} raw candidates ({arxiv_query_number} arXiv queries)')
    return {'raw_candidates': raw_candidates, 'uploaded_s2_paper': uploaded_s2_paper,
            'discovery_summary': result['answer']}


def uploaded_paper_record(state: PaperState) -> dict:
    """The uploaded paper in candidate format, used to exclude it from the results."""
    s2 = state.get('uploaded_s2_paper') or {}
    record = candidate_from_s2(s2, 'upload') if s2 else make_candidate(state['profile']['title'], '', [], None, '', 'upload')
    record['arxiv_id'] = record['arxiv_id'] or state['identifiers'].get('arxiv_id', '')
    record['doi'] = record['doi'] or clean_doi(state['identifiers'].get('doi', ''))
    return record


@traced('merge_dedup')
async def merge_dedup_node(state: PaperState) -> dict:
    candidates, stats = merge_and_deduplicate(state['raw_candidates'], uploaded_paper_record(state))
    # also exclude by the profile title (in case Semantic Scholar did not find the paper)
    profile_title = normalize_title(state['profile']['title'])
    candidates = [c for c in candidates if normalize_title(c['title']) != profile_title]
    stats['unique_usable'] = len(candidates)
    trace_note(f'merge stats: {stats}')
    return {'candidates': candidates, 'merge_stats': stats}


@traced('score_rank')
async def score_rank_node(state: PaperState) -> dict:
    s2_abstract = (state.get('uploaded_s2_paper') or {}).get('abstract')
    profile = state['profile']
    uploaded_text = f"{profile['title']}. {s2_abstract or profile['abstract']}"
    ranked = await asyncio.to_thread(score_and_rank, uploaded_text, profile, state['candidates'])
    return {'ranked_papers': ranked}


@traced('explain_ranking')
async def explain_ranking_node(state: PaperState) -> dict:
    ranked = [dict(paper) for paper in state['ranked_papers']]
    for paper in ranked[:TOP_N_RESULTS]:
        paper['explanation'] = await asyncio.to_thread(explain_ranking, paper, state['profile'])
    return {'ranked_papers': ranked}


@traced('paper_rag_agent')
async def paper_rag_agent_node(state: PaperState) -> dict:
    thread_id = os.path.basename(state['pdf_path'])           # one conversation memory per paper
    result = await run_agent(paper_rag_agent, state['question'], thread_id=thread_id, max_steps=8)
    return {'answer': result['answer']}


print('State and nodes defined.')

## 17. Graph Routing

Three router functions decide where the graph goes next. They only read the state, so they are easy to test.

In [ ]:
def route_request(state: PaperState) -> str:
    """START: process the paper on the first run; afterwards go straight to the RAG agent."""
    if not state.get('rag_ready'):
        return 'load_pdf'
    return 'paper_rag_agent' if state.get('question') else END


def route_after_index(state: PaperState) -> str:
    """After indexing: answer the question if one was given, otherwise this branch ends."""
    return 'paper_rag_agent' if state.get('question') else END


def route_after_merge(state: PaperState) -> str:
    """If discovery found nothing usable (e.g. APIs down), skip scoring gracefully."""
    return 'score_rank' if state.get('candidates') else END


# Router unit checks
assert route_request({}) == 'load_pdf'
assert route_request({'rag_ready': True, 'question': 'What is the method?'}) == 'paper_rag_agent'
assert route_after_index({'question': ''}) == END
assert route_after_merge({'candidates': []}) == END
print('Routers behave as expected.')

## 18. Build and Compile the Graph

In [ ]:
graph = StateGraph(PaperState)

graph.add_node('load_pdf', load_pdf_node)
graph.add_node('extract_profile', extract_profile_node)
graph.add_node('build_rag_index', build_rag_index_node)
graph.add_node('discovery_agent', discovery_agent_node)
graph.add_node('merge_dedup', merge_dedup_node)
graph.add_node('score_rank', score_rank_node)
graph.add_node('explain_ranking', explain_ranking_node)
graph.add_node('paper_rag_agent', paper_rag_agent_node)

graph.add_conditional_edges(START, route_request, ['load_pdf', 'paper_rag_agent', END])
graph.add_edge('load_pdf', 'extract_profile')
graph.add_edge('extract_profile', 'build_rag_index')     # parallel fan-out:
graph.add_edge('extract_profile', 'discovery_agent')     # both run in the same step
graph.add_conditional_edges('build_rag_index', route_after_index, ['paper_rag_agent', END])
graph.add_edge('discovery_agent', 'merge_dedup')
graph.add_conditional_edges('merge_dedup', route_after_merge, ['score_rank', END])
graph.add_edge('score_rank', 'explain_ranking')
graph.add_edge('explain_ranking', END)
graph.add_edge('paper_rag_agent', END)

paper_graph = graph.compile()

try:
    from IPython.display import Image, display
    display(Image(paper_graph.get_graph().draw_mermaid_png()))
except Exception:
    print(paper_graph.get_graph().draw_mermaid())       # text version if the image service is unreachable

## 19. Run: Process the Paper + Discover Similar Papers

One call runs the whole first pass: load, profile, then **in parallel** index the paper and run Agent 2
with merge, score, and explain. We also pass a first question, so Agent 1 answers it as soon as the index is ready.

Watch the **trace tree** printed at the end: it shows every node, agent, tool call, and LLM call, with durations and tokens.
This takes about 1–3 minutes (arXiv asks clients to wait 3 seconds between requests).

In [ ]:
@traced('analyze_paper', 'run')
async def analyze_paper(pdf_path: str, question: str = '') -> dict:
    rag_loop_detector.reset()
    return await paper_graph.ainvoke({'pdf_path': pdf_path, 'question': question})


paper_state = await analyze_paper(PDF_PATH, question='What is the main objective of this paper?')

print('PROFILE')
print(json.dumps({k: v for k, v in paper_state['profile'].items() if k != 'abstract'}, indent=2))
print('\nDiscovery agent summary:\n', paper_state.get('discovery_summary', ''))
print('\nMerge stats:', paper_state.get('merge_stats'))
print('\nFirst answer:\n', paper_state.get('answer', ''))

## 20. Display Similar Papers

In [ ]:
import pandas as pd
from IPython.display import Markdown, display


def format_authors(authors: list, max_authors: int = 4) -> str:
    if not authors:
        return 'unknown'
    return ', '.join(authors[:max_authors]) + (' et al.' if len(authors) > max_authors else '')


def show_ranked_papers(state: PaperState, top_n: int = TOP_N_RESULTS):
    ranked = state.get('ranked_papers') or []
    if not ranked:
        display(Markdown('**No related papers could be ranked** (discovery returned no usable candidates).'))
        return
    blocks = [f"## Top {min(top_n, len(ranked))} papers related to: *{state['profile']['title']}*\n"
              f"<sub>{state['merge_stats']['unique_usable']} unique candidates were scored.</sub>"]
    for paper in ranked[:top_n]:
        c = paper['components']
        blocks.append(
            f"### {paper['rank']}. {paper['title']}\n"
            f"**Authors:** {format_authors(paper['authors'])}  \n"
            f"**Year:** {paper.get('year') or 'n/a'} &nbsp;|&nbsp; **URL:** {paper.get('url') or 'n/a'}  \n"
            f"**Discovery sources:** {', '.join(paper['found_in'])}  \n"
            + (f"**Citation relationship:** {paper['citation_relationship']}  \n" if paper['citation_relationship'] else '')
            + f"**Overall Relatedness Score: {paper['overall_relatedness']:.1f}%** &nbsp;|&nbsp; "
            f"**Semantic Similarity Score: {100 * c['semantic_similarity']:.1f}%**  \n"
            f"<sub>components: semantic {c['semantic_similarity']:.3f} · citation {c['citation_relationship']:.0f} · "
            f"agreement {c['multi_source_agreement']:.1f} · topic overlap {c['topic_overlap']:.2f}</sub>\n\n"
            f"**Why this paper ranked highly:**\n\n{paper.get('explanation', '')}\n\n---"
        )
    display(Markdown('\n\n'.join(blocks)))


def ranking_table(state: PaperState, top_n: int = TOP_N_RESULTS) -> pd.DataFrame:
    rows = [{'rank': p['rank'], 'title': p['title'][:60], 'year': p.get('year'),
             'overall_%': p['overall_relatedness'], 'semantic_%': round(100 * p['components']['semantic_similarity'], 1),
             'citation': p['components']['citation_relationship'], 'agreement': p['components']['multi_source_agreement'],
             'topic': p['components']['topic_overlap'], 'sources': ', '.join(p['found_in'])}
            for p in (state.get('ranked_papers') or [])[:top_n]]
    return pd.DataFrame(rows)


display(ranking_table(paper_state))
show_ranked_papers(paper_state)

## 21. Ask Questions About the Paper

Later questions go through the **same graph**. Because `rag_ready` is already `True`, the START router
sends them **straight to the Paper RAG Agent**, so nothing is re-processed. Thanks to the checkpointer, the agent
remembers earlier questions, so follow-ups work.

In [ ]:
@traced('ask_question', 'run')
async def ask_question(question: str) -> str:
    global paper_state
    rag_loop_detector.reset()
    paper_state = await paper_graph.ainvoke({**paper_state, 'question': question})
    return paper_state['answer']


async def ask(question: str):
    answer = await ask_question(question)
    display(Markdown(f'**Q: {question}**\n\n{answer}'))


await ask('What methodology was used?')

In [ ]:
await ask('What datasets or benchmarks were used for evaluation?')

In [ ]:
await ask('What are the main findings?')

In [ ]:
await ask('What limitations are mentioned?')

In [ ]:
# Follow-up question: uses the agent's memory (checkpointer) of the previous turn
await ask('Explain those limitations in simpler words, in two sentences.')

In [ ]:
# Ask your own question here
await ask('Summarize the paper in 5 bullet points.')

## 22. Final Tests

Automatic checks against the acceptance criteria, using the real run above.

In [ ]:
def check(condition: bool, label: str):
    print(('PASS ' if condition else 'FAIL ') + label)
    return condition


ranked = paper_state.get('ranked_papers') or []
top = ranked[:TOP_N_RESULTS]
results = [
    check(len(paper_state['pages']) > 0, '1. PDF uploaded and processed'),
    check(PAPER_INDEX['num_chunks'] > 0, f"2. RAG index built ({PAPER_INDEX['num_chunks']} chunks)"),
    check(bool(paper_state.get('answer')), '3. Paper RAG Agent answered questions'),
    check(any(s.startswith('semantic_scholar') for c in paper_state['raw_candidates'] for s in c['found_in']),
          '4. Semantic Scholar provided candidates'),
    check(any(s.startswith('arxiv') for c in paper_state['raw_candidates'] for s in c['found_in']),
          '5. arXiv provided candidates'),
]

keys_seen = [key for paper in ranked for key in identity_keys(paper)]
results.append(check(len(keys_seen) == len(set(keys_seen)), '6. Candidates merged and deduplicated (no shared identifiers)'))
uploaded_title = normalize_title(paper_state['profile']['title'])
results.append(check(all(normalize_title(p['title']) != uploaded_title for p in ranked), '6b. Uploaded paper excluded'))

recomputed_ok = all(abs(overall_relatedness(p['components']) - p['overall_relatedness']) < 1e-9 for p in ranked)
results.append(check(recomputed_ok and all(0 <= p['overall_relatedness'] <= 100 for p in ranked),
                     '7. Scores computed in Python (recomputed from components, all within 0-100)'))
scores = [p['overall_relatedness'] for p in ranked]
results.append(check(scores == sorted(scores, reverse=True), '8. Ranked from highest to lowest'))
results.append(check(all(p.get('explanation') for p in top), f'9. All top {len(top)} papers have explanations'))
results.append(check(len(AGENTS) == 2, '10. Exactly two agents: ' + ', '.join(a.name for a in AGENTS)))
results.append(check(isinstance(paper_graph, type(graph.compile())), '11. LangGraph orchestrates the system'))
conditional = [e for e in paper_graph.get_graph().edges if e.conditional]
results.append(check(len(conditional) >= 3, f'12. Graph has conditional routing ({len(conditional)} conditional edges) + parallel fan-out'))
print(f'\n{sum(results)}/{len(results)} checks passed')

In [ ]:
# Grounding test: a question the paper cannot answer. Expect "I could not find this in the uploaded paper."
await ask("What is the first author's favourite food?")

## 23. Architecture Explanation (for the presentation)

### The two agents
| | Paper RAG Agent | Discovery & Ranking Agent |
|---|---|---|
| Built with | `create_agent` | `create_agent` |
| Trigger | a user question | automatically, when a paper is processed |
| LLM decides | what to search in the paper, when it has enough | which identifiers to use, which 4–5 arXiv queries to write |
| Tools | `search_paper` | `get_semantic_scholar_candidates`, `search_arxiv` |
| Limits | 4 tool calls / question, loop detector | 8 tool calls / run, loop + stagnation detector |
| Memory | checkpointer (thread per paper) | none needed (one-shot task) |

### Nodes (not agents)
| Node | Type | What it does |
|---|---|---|
| `load_pdf` | Python | PDF → pages with page numbers, finds arXiv ID / DOI |
| `extract_profile` | 1 structured LLM call | title, abstract, problem, method, domain, concepts, keywords |
| `build_rag_index` | Python | chunks → embeddings → FAISS |
| `discovery_agent` | **Agent 2** | collects candidates through its tools |
| `merge_dedup` | Python | dedup by S2 ID → arXiv ID → DOI → title; exclusions |
| `score_rank` | Python | cosine similarity + weighted formula, sort |
| `explain_ranking` | 1 LLM call per top paper | explanation from evidence only |
| `paper_rag_agent` | **Agent 1** | grounded answer with page citations |

### Key design decisions
1. **Numbers come from Python, words come from the LLM.** The LLM never computes or edits a score.
   The final test recomputes every score from its components.
2. **Artifacts instead of copied data.** Tools give the LLM short summaries, while the full paper metadata goes
   through tool artifacts straight to Python, so the LLM cannot corrupt titles or IDs.
3. **Routing instead of a chain.** The START router reuses the processed paper for questions;
   the fan-out runs indexing and discovery in parallel; the merge router stops gracefully if no candidates were found.
4. **Reliability.** Tool-call limits, repetition + stagnation detection with visible warnings,
   Semantic Scholar retries on 429, and arXiv rate-limit compliance.
5. **Observability.** Our tracer prints one tree per run with every node, agent, tool, and LLM call, plus durations and tokens.

### Limitations
- MiniLM embeddings are general-purpose. A scientific model (e.g. SPECTER2) could give sharper similarity.
- Semantic Scholar sometimes hides abstracts for publisher papers, and those candidates are excluded.
- Scores are **relative relatedness**, not accuracy or probability. Comparing papers by score is meaningful only within one run.